# 02 — EW-DDBS pada kohort CTU-UHB (PhysioNet) — v3.2

**Tujuan run ini: menambahkan seed 7 dan 2024.** Seed 42 dan kontrol
permutasi SUDAH selesai dan hasilnya sudah dipakai di naskah — jangan
dijalankan ulang (lihat Sel 5).

Perubahan dari v3.1:

1. Sel versi pustaka dipindah ke **setelah** `pip install` (sebelumnya
   mengimpor `imblearn`/`xgboost` sebelum keduanya terpasang).
2. Sel pemulihan cache fitur — menghindari unduh ulang 552 rekaman
   PhysioNet (~25 menit) setiap kali runtime di-reset.
3. Sel seed 42 kini **dijaga**: berhenti bila `results_ctu/` sudah ada.
4. Sel baru untuk **seed 7** dan **seed 2024**.
5. `merge_runs.py` ditambahkan — sebelumnya tidak ada di notebook CTU.
6. Sel unduh kini mencakup `results_ctu_s3` (sebelumnya terlewat).

> GPU: Runtime > Change runtime type > **T4**, sama seperti seed 42.

## 0. Dependensi

In [ ]:
# Dependensi (TensorFlow sudah tersedia di Colab)
!pip install -q PyWavelets wfdb imbalanced-learn xgboost scikit-posthocs xlrd
import tensorflow as tf
print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU') or 'TIDAK ADA - aktifkan di Runtime > Change runtime type')

In [ ]:
import sys, tensorflow, sklearn, imblearn, xgboost, numpy, pandas, scipy
for m in (sys, tensorflow, sklearn, imblearn, xgboost, numpy, pandas, scipy):
    print(getattr(m, '__name__', 'python'), getattr(m, '__version__', sys.version.split()[0]))

## 1. Cache fitur 62 dimensi

Ekstraksi 552 rekaman memakan ~25 menit. Bila Anda punya
`ctu_uhb_features62_cache.npz`, unggah lewat panel **Files** sebelum
menjalankan sel ini.

In [ ]:
import os, shutil

CACHE = 'ctu_uhb_features62_cache.npz'
DRIVE = '/content/drive/MyDrive/ewddbs'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE, exist_ok=True)
except Exception as e:
    print('Drive tidak di-mount:', e)

if not os.path.exists(CACHE) and os.path.exists(f'{DRIVE}/{CACHE}'):
    shutil.copy(f'{DRIVE}/{CACHE}', CACHE)
    print('cache dipulihkan dari Drive')

if os.path.exists(CACHE):
    import numpy as np
    d = np.load(CACHE, allow_pickle=True)
    print(f'cache OK: {d["X"].shape[0]} sampel x {d["X"].shape[1]} fitur')
    assert d['X'].shape[1] == 62, 'cache bukan 62 fitur - hapus dan bangun ulang'
else:
    print('cache TIDAK ADA - run pertama akan mengunduh PhysioNet (~25 menit)')

## 2. Tulis modul ke disk

In [ ]:
%%writefile ctg_features.py
"""
ctg_features.py
===============
62-dimensional feature extractor for raw CTU-UHB cardiotocography signals.

Implements EXACTLY the feature families described in Contribution (2) of the
manuscript:

  (a) basic descriptive statistics                       -> 22 (11 x 2 channels)
  (b) wavelet decomposition (Daubechies db4, 3 levels)   ->  8 ( 4 x 2 channels)
  (c) multi-band spectral analysis (VLF/LF/HF + SEF95)   -> 10 ( 5 x 2 channels)
  (d) nonlinear entropy (Sample, Approximate, Permutation)->  6 ( 3 x 2 channels)
  (e) CTG-specific morphological descriptors             -> 11 (8 FHR + 3 UC)
  (f) cross-channel FHR-UC coupling                      ->  5
                                                        -------
                                                    TOTAL   62

Signals are sampled at 4 Hz in CTU-UHB.

Author note: every returned value is finite; NaN/Inf are replaced by 0.0 so the
downstream imputer never has to guess.
"""

import math

import numpy as np
import pywt
from scipy import stats, signal as sps

FS = 4.0  # CTU-UHB sampling rate (Hz)


# ----------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------
def _clean(x, lo=None, hi=None, max_gap_s=15.0, fs=FS):
    """Remove non-physiological values and linearly interpolate short gaps."""
    x = np.asarray(x, dtype=float).copy()
    if lo is not None:
        x[x < lo] = np.nan
    if hi is not None:
        x[x > hi] = np.nan
    x[x == 0] = np.nan  # CTU-UHB codes signal loss as 0

    n = len(x)
    if n == 0:
        return x
    idx = np.arange(n)
    good = ~np.isnan(x)
    if good.sum() < 10:
        return np.full(n, np.nan)

    # interpolate gaps shorter than max_gap_s, leave long gaps as NaN
    xi = np.interp(idx, idx[good], x[good])
    max_gap = int(max_gap_s * fs)
    isnan = np.isnan(x)
    if isnan.any():
        # find runs of NaN
        d = np.diff(np.concatenate([[0], isnan.view(np.int8), [0]]))
        starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]
        for s, e in zip(starts, ends):
            if (e - s) > max_gap:
                xi[s:e] = np.nan
    return xi


def _finite(vals):
    out = []
    for v in vals:
        v = float(v) if v is not None else 0.0
        out.append(v if np.isfinite(v) else 0.0)
    return out


def _downsample(x, target_n=2000):
    """Decimate for O(N^2) entropy estimators."""
    x = x[np.isfinite(x)]
    if len(x) <= target_n:
        return x
    step = int(np.ceil(len(x) / target_n))
    return x[::step]


# ----------------------------------------------------------------------
# (a) descriptive statistics -- 11 per channel
# ----------------------------------------------------------------------
def descriptive_features(x):
    v = x[np.isfinite(x)]
    if len(v) < 10:
        return _finite([0] * 11)
    q25, q75 = np.percentile(v, [25, 75])
    hist, _ = np.histogram(v, bins=10, density=True)
    hist = hist[hist > 0]
    hist_ent = -np.sum(hist * np.log(hist + 1e-12))
    return _finite([
        np.mean(v), np.std(v), np.median(v), np.min(v), np.max(v),
        np.max(v) - np.min(v), q75 - q25, np.percentile(v, 90),
        stats.skew(v), stats.kurtosis(v), hist_ent,
    ])


# ----------------------------------------------------------------------
# (b) wavelet db4, 3 levels -- 4 per channel (log energy of cA3,cD3,cD2,cD1)
# ----------------------------------------------------------------------
def wavelet_features(x, wavelet='db4', level=3):
    v = x[np.isfinite(x)]
    if len(v) < 2 ** (level + 2):
        return _finite([0] * 4)
    try:
        coeffs = pywt.wavedec(v, wavelet, level=level)   # [cA3, cD3, cD2, cD1]
    except Exception:
        return _finite([0] * 4)
    return _finite([np.log10(np.sum(c ** 2) + 1e-12) for c in coeffs])


# ----------------------------------------------------------------------
# (c) multi-band spectral -- 5 per channel
# ----------------------------------------------------------------------
def spectral_features(x, fs=FS):
    v = x[np.isfinite(x)]
    if len(v) < 256:
        return _finite([0] * 5)
    nper = int(min(1024, len(v)))
    f, psd = sps.welch(v - np.mean(v), fs=fs, nperseg=nper)
    if len(f) < 2:
        return _finite([0] * 5)
    df = f[1] - f[0]

    def band(lo, hi):
        m = (f >= lo) & (f < hi)
        return float(np.sum(psd[m]) * df)

    vlf = band(0.0, 0.03)     # very low frequency
    lf = band(0.03, 0.15)     # low frequency
    hf = band(0.15, 0.50)     # high frequency
    lf_hf = lf / (hf + 1e-12)

    # spectral edge frequency 95%
    total = np.cumsum(psd) * df
    if total[-1] <= 0:
        sef95 = 0.0
    else:
        sef95 = float(f[np.searchsorted(total, 0.95 * total[-1])])

    return _finite([np.log10(vlf + 1e-12), np.log10(lf + 1e-12),
                    np.log10(hf + 1e-12), lf_hf, sef95])


# ----------------------------------------------------------------------
# (d) nonlinear entropies -- 3 per channel
# ----------------------------------------------------------------------
def _phi(x, m, r):
    n = len(x)
    if n <= m + 1:
        return None
    emb = np.lib.stride_tricks.sliding_window_view(x, m)[: n - m + 1]
    # chunked Chebyshev distance to keep memory bounded
    counts = np.zeros(len(emb))
    chunk = 512
    for i in range(0, len(emb), chunk):
        d = np.max(np.abs(emb[i:i + chunk, None, :] - emb[None, :, :]), axis=2)
        counts[i:i + chunk] = np.sum(d <= r, axis=1)
    return counts


def approximate_entropy(x, m=2, r_factor=0.2):
    x = _downsample(x, 1200)
    if len(x) < 50:
        return 0.0
    r = r_factor * np.std(x)
    if r <= 0:
        return 0.0
    c_m = _phi(x, m, r)
    c_m1 = _phi(x, m + 1, r)
    if c_m is None or c_m1 is None:
        return 0.0
    phi_m = np.mean(np.log(c_m / len(c_m) + 1e-12))
    phi_m1 = np.mean(np.log(c_m1 / len(c_m1) + 1e-12))
    return float(phi_m - phi_m1)


def sample_entropy(x, m=2, r_factor=0.2):
    x = _downsample(x, 1200)
    if len(x) < 50:
        return 0.0
    r = r_factor * np.std(x)
    if r <= 0:
        return 0.0
    c_m = _phi(x, m, r)
    c_m1 = _phi(x, m + 1, r)
    if c_m is None or c_m1 is None:
        return 0.0
    # exclude self-matches
    A = np.sum(c_m1 - 1)
    B = np.sum(c_m[: len(c_m1)] - 1)
    if A <= 0 or B <= 0:
        return 0.0
    return float(-np.log(A / B))


def permutation_entropy(x, order=3, delay=1, normalise=True):
    x = _downsample(x, 5000)
    n = len(x)
    if n < order * delay + 1:
        return 0.0
    emb = np.array([x[i:i + (order - 1) * delay + 1:delay]
                    for i in range(n - (order - 1) * delay)])
    patterns = np.argsort(emb, axis=1)
    _, counts = np.unique(patterns, axis=0, return_counts=True)
    p = counts / counts.sum()
    pe = -np.sum(p * np.log2(p))
    if normalise:
        pe /= np.log2(math.factorial(order))
    return float(pe)


def entropy_features(x):
    return _finite([sample_entropy(x), approximate_entropy(x),
                    permutation_entropy(x)])


# ----------------------------------------------------------------------
# (e) CTG-specific morphology
# ----------------------------------------------------------------------
def fhr_morphology(fhr, fs=FS):
    """8 features: baseline, STV, LTV, %abnormal STV, accel, decel,
    prolonged decel, baseline drift."""
    v = fhr.copy()
    good = np.isfinite(v)
    if good.sum() < int(60 * fs):
        return _finite([0] * 8)
    vv = v[good]

    # baseline = mode of 5-bpm-binned histogram (clinical convention)
    bins = np.arange(np.floor(vv.min()), np.ceil(vv.max()) + 5, 5)
    if len(bins) < 2:
        baseline = float(np.median(vv))
    else:
        h, edges = np.histogram(vv, bins=bins)
        baseline = float((edges[np.argmax(h)] + edges[np.argmax(h) + 1]) / 2)

    # short-term variation: mean |diff| over 3.75 s epochs (Dawes-Redman style)
    ep = int(3.75 * fs)
    n_ep = len(vv) // ep
    if n_ep >= 2:
        epochs = vv[:n_ep * ep].reshape(n_ep, ep).mean(axis=1)
        stv = float(np.mean(np.abs(np.diff(epochs))))
        # long-term variation: mean range per 1-min window
        per_min = int(60 * fs / ep)
        if n_ep >= per_min and per_min > 0:
            nw = n_ep // per_min
            w = epochs[:nw * per_min].reshape(nw, per_min)
            ltv = float(np.mean(w.max(axis=1) - w.min(axis=1)))
        else:
            ltv = float(np.ptp(epochs))
        pct_abn_stv = float(np.mean(np.abs(np.diff(epochs)) < 1.0) * 100)
    else:
        stv, ltv, pct_abn_stv = 0.0, 0.0, 0.0

    # accelerations: >= +15 bpm above baseline for >= 15 s
    # decelerations : <= -15 bpm below baseline for >= 15 s
    # prolonged dec.: <= -15 bpm for >= 120 s
    def _episodes(mask, min_s):
        m = mask.astype(np.int8)
        d = np.diff(np.concatenate([[0], m, [0]]))
        st, en = np.where(d == 1)[0], np.where(d == -1)[0]
        dur = (en - st) / fs
        return int(np.sum(dur >= min_s))

    above = np.nan_to_num(v, nan=baseline) >= baseline + 15
    below = np.nan_to_num(v, nan=baseline) <= baseline - 15
    accel = _episodes(above, 15)
    decel = _episodes(below, 15)
    prol = _episodes(below, 120)

    # baseline drift: slope of a linear fit over the whole trace (bpm/hour)
    t = np.arange(len(v))[good] / fs / 3600.0
    if len(t) > 2 and np.ptp(t) > 0:
        drift = float(np.polyfit(t, vv, 1)[0])
    else:
        drift = 0.0

    return _finite([baseline, stv, ltv, pct_abn_stv,
                    accel, decel, prol, drift])


def uc_morphology(uc, fs=FS):
    """3 features: contraction count, mean amplitude, mean duration."""
    v = uc[np.isfinite(uc)]
    if len(v) < int(60 * fs):
        return _finite([0] * 3)
    base = np.percentile(v, 25)
    thr = base + 0.5 * (np.percentile(v, 90) - base)
    m = (np.nan_to_num(uc, nan=base) > thr).astype(np.int8)
    d = np.diff(np.concatenate([[0], m, [0]]))
    st, en = np.where(d == 1)[0], np.where(d == -1)[0]
    dur = (en - st) / fs
    keep = dur >= 30.0                      # a contraction lasts >= 30 s
    if keep.sum() == 0:
        return _finite([0, 0, 0])
    amps = [np.nanmax(uc[s:e]) - base for s, e in zip(st[keep], en[keep])]
    return _finite([int(keep.sum()), np.mean(amps), np.mean(dur[keep])])


# ----------------------------------------------------------------------
# (f) cross-channel FHR-UC coupling -- 5
# ----------------------------------------------------------------------
def coupling_features(fhr, uc, fs=FS):
    a, b = fhr.copy(), uc.copy()
    good = np.isfinite(a) & np.isfinite(b)
    if good.sum() < int(120 * fs):
        return _finite([0] * 5)
    a, b = a[good], b[good]
    a = (a - a.mean()) / (a.std() + 1e-12)
    b = (b - b.mean()) / (b.std() + 1e-12)

    pearson = float(np.corrcoef(a, b)[0, 1])

    # cross-correlation within +/- 120 s
    maxlag = int(120 * fs)
    n = len(a)
    lags = np.arange(-maxlag, maxlag + 1)
    xc = np.correlate(a, b, mode='full') / n
    centre = n - 1
    seg = xc[centre - maxlag: centre + maxlag + 1] if n > maxlag else xc
    if len(seg) == 0:
        max_xc, lag_at_max = 0.0, 0.0
    else:
        k = int(np.argmax(np.abs(seg)))
        max_xc = float(seg[k])
        lag_at_max = float(lags[k] / fs) if len(seg) == len(lags) else 0.0

    # magnitude-squared coherence in the LF band
    try:
        f, cxy = sps.coherence(a, b, fs=fs, nperseg=int(min(1024, n)))
        m = (f >= 0.03) & (f < 0.15)
        lf_coh = float(np.mean(cxy[m])) if m.any() else 0.0
    except Exception:
        lf_coh = 0.0

    # fraction of FHR decelerations that overlap a contraction
    dec = a < -1.0
    con = b > 1.0
    overlap = float(np.sum(dec & con) / (np.sum(dec) + 1e-12))

    return _finite([pearson, max_xc, lag_at_max, lf_coh, overlap])


# ----------------------------------------------------------------------
# public API
# ----------------------------------------------------------------------
FEATURE_NAMES = (
    [f'FHR_{n}' for n in ['mean', 'std', 'median', 'min', 'max', 'range',
                          'iqr', 'p90', 'skew', 'kurt', 'hist_ent']] +
    [f'UC_{n}' for n in ['mean', 'std', 'median', 'min', 'max', 'range',
                         'iqr', 'p90', 'skew', 'kurt', 'hist_ent']] +
    [f'FHR_wav_{n}' for n in ['cA3', 'cD3', 'cD2', 'cD1']] +
    [f'UC_wav_{n}' for n in ['cA3', 'cD3', 'cD2', 'cD1']] +
    [f'FHR_{n}' for n in ['VLF', 'LF', 'HF', 'LFHF', 'SEF95']] +
    [f'UC_{n}' for n in ['VLF', 'LF', 'HF', 'LFHF', 'SEF95']] +
    [f'FHR_{n}' for n in ['SampEn', 'ApEn', 'PermEn']] +
    [f'UC_{n}' for n in ['SampEn', 'ApEn', 'PermEn']] +
    ['FHR_baseline', 'FHR_STV', 'FHR_LTV', 'FHR_pctAbnSTV',
     'FHR_accel_n', 'FHR_decel_n', 'FHR_prolDecel_n', 'FHR_drift'] +
    ['UC_contraction_n', 'UC_mean_amp', 'UC_mean_dur'] +
    ['XC_pearson', 'XC_maxcorr', 'XC_lag_s', 'XC_lf_coherence',
     'XC_decel_uc_overlap']
)
assert len(FEATURE_NAMES) == 62, f'expected 62 names, got {len(FEATURE_NAMES)}'


def extract_62_features(fhr_raw, uc_raw, fs=FS):
    """Return a 62-vector of features for one CTU-UHB recording."""
    fhr = _clean(fhr_raw, lo=50, hi=200, fs=fs)
    uc = _clean(uc_raw, lo=0, hi=100, fs=fs)

    feats = []
    feats += descriptive_features(fhr)      # 11
    feats += descriptive_features(uc)       # 11  -> 22
    feats += wavelet_features(fhr)          #  4
    feats += wavelet_features(uc)           #  4  -> 30
    feats += spectral_features(fhr, fs)     #  5
    feats += spectral_features(uc, fs)      #  5  -> 40
    feats += entropy_features(fhr)          #  3
    feats += entropy_features(uc)           #  3  -> 46
    feats += fhr_morphology(fhr, fs)        #  8  -> 54
    feats += uc_morphology(uc, fs)          #  3  -> 57
    feats += coupling_features(fhr, uc, fs) #  5  -> 62

    assert len(feats) == 62, f'expected 62 features, got {len(feats)}'
    return np.asarray(feats, dtype=np.float32)


if __name__ == '__main__':
    rng = np.random.default_rng(0)
    n = 19200
    t = np.arange(n) / FS
    fhr = 140 + 8 * np.sin(2 * np.pi * 0.02 * t) + rng.normal(0, 3, n)
    fhr[5000:5200] -= 30          # a deceleration
    fhr[9000:9600] -= 25          # a prolonged deceleration
    uc = 20 + 30 * (np.sin(2 * np.pi * t / 180) > 0.7) + rng.normal(0, 2, n)
    fhr[100:400] = 0              # simulated signal loss

    f = extract_62_features(fhr, uc)
    print('n features :', len(f))
    print('all finite :', bool(np.all(np.isfinite(f))))
    print('n names    :', len(FEATURE_NAMES))
    for name, val in list(zip(FEATURE_NAMES, f))[:6]:
        print(f'  {name:24s} {val: .4f}')
    print('  ...')
    for name, val in list(zip(FEATURE_NAMES, f))[-8:]:
        print(f'  {name:24s} {val: .4f}')

In [ ]:
%%writefile ewddbs_core.py
"""
ewddbs_core.py
==============
Shared implementation of EW-DDBS, metrics, baselines and artefact logging.

Fixes applied relative to the original notebooks (see audit):
  D6  neighbour search is performed in the LATENT space on BOTH cohorts
  D7  the ablation grid genuinely isolates the entropy term
      (use_entropy and use_safe are independent switches, 2x2x2 = 8 variants)
  D8  jitter covariance follows Eq. (6): diagonal, eta * Var(minority-only)
  D9  a single G-Mean definition is used everywhere
  T1  y_proba is persisted so calibration (Brier / reliability) can be computed
      without re-running the experiment
  T2  H(x), S(z) and W_i are persisted so the weighting scheme can be shown to
      be non-degenerate
  T3  Baseline (no resampling), ClassWeight and PriorCorrection baselines added
  S2  every run is seeded and the seed is recorded with each result row
"""

import json
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.discriminant_analysis import (LinearDiscriminantAnalysis,
                                           QuadraticDiscriminantAnalysis)
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                              ExtraTreesClassifier, GradientBoostingClassifier,
                              RandomForestClassifier)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             brier_score_loss, confusion_matrix, f1_score,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import label_binarize
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from imblearn.under_sampling import TomekLinks
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

# TensorFlow is imported lazily so that the post-hoc analysis script can use
# this module without pulling in the deep-learning stack.
tf = None


def _load_tf():
    global tf
    if tf is None:
        import tensorflow as _tf
        tf = _tf
    return tf

# ----------------------------------------------------------------------
# hyper-parameters (identical on both cohorts -- see Table 2 correction)
# ----------------------------------------------------------------------
T_TEMP = 0.5      # entropy temperature
TAU = 0.2         # safe-region threshold
K_NN = 5          # neighbourhood size for the Safe-Ratio
ETA_JITTER = 0.05 # jitter scale (eta in Eq. 6)


def set_seed(seed):
    np.random.seed(seed)
    _load_tf().random.set_seed(seed)


# ======================================================================
# 1. CNN topology guide
# ======================================================================
def cnn_topology_guide(X_train, y_train, seed=42, epochs=100, patience=5):
    """Train the 1D-CNN guide and return (entropy, latent_features).

    NOTE: entropy and latent codes are computed IN-SAMPLE, exactly as in the
    manuscript. The diagnostics returned by ew_ddbs_resample() are what reveal
    whether this makes the weighting degenerate.
    """
    _load_tf()
    from tensorflow.keras import callbacks, layers, models

    tf.keras.backend.clear_session()
    set_seed(seed)
    n_features = X_train.shape[1]
    n_classes = len(np.unique(y_train))

    inputs = layers.Input(shape=(n_features, 1))
    x = layers.Conv1D(64, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(32, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    latent = layers.GlobalAveragePooling1D()(x)          # 32-D latent vector
    outputs = layers.Dense(n_classes, activation='softmax')(latent)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy')

    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    cw_dict = {int(c): float(w) for c, w in zip(np.unique(y_train), cw)}

    X_cnn = X_train.reshape(-1, n_features, 1)
    es = callbacks.EarlyStopping(monitor='loss', patience=patience,
                                 restore_best_weights=True)
    model.fit(X_cnn, y_train, epochs=epochs, batch_size=32,
              class_weight=cw_dict, verbose=0, callbacks=[es])

    probs = model.predict(X_cnn, verbose=0)
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
    extractor = models.Model(inputs, latent)
    deep_feats = extractor.predict(X_cnn, verbose=0)
    return entropy.astype(np.float32), deep_feats.astype(np.float32)


# ======================================================================
# 2. EW-DDBS
# ======================================================================
def ew_ddbs_resample(X, y, entropy, latent,
                     use_entropy=True, use_safe=True, apply_tomek=True,
                     T=T_TEMP, tau=TAU, k=K_NN, eta=ETA_JITTER, rng=None):
    """Latent-guided safe-region oversampling.

    W_i = exp(H(x_i)/T) * S(z_i)      (Eq. 4)   -- both terms optional
    x_new = x_p + lam*(x_n - x_p) + d (Eq. 5)
    d ~ N(0, eta * diag(Var_minority)) (Eq. 6)

    Parents are drawn with P(i) ~ W_i; the neighbour x_n is the nearest
    same-class neighbour IN THE LATENT SPACE; synthesis happens in input space.

    Returns (X_res, y_res, diagnostics).
    """
    rng = np.random.default_rng(0) if rng is None else rng
    X_res, y_res = X.copy(), y.copy()
    classes, counts = np.unique(y, return_counts=True)
    max_count = counts.max()
    diag = {}

    for cls in classes:
        mask = y == cls
        n_cls = int(mask.sum())
        if n_cls >= max_count:
            continue
        num_add = int(max_count - n_cls)
        idx_cls = np.where(mask)[0]

        if n_cls < 2:
            chosen = rng.choice(idx_cls, size=num_add, replace=True)
            X_res = np.vstack([X_res, X[chosen]])
            y_res = np.hstack([y_res, np.full(num_add, cls)])
            continue

        # ---- Safe-Ratio in the latent manifold (Eq. 3) -------------------
        k_eff = int(min(k, len(latent) - 1))
        nn_lat_all = NearestNeighbors(n_neighbors=k_eff + 1).fit(latent)
        _, ind_all = nn_lat_all.kneighbors(latent[idx_cls])
        safe_ratio = np.mean(y[ind_all[:, 1:]] == cls, axis=1)

        # ---- weights (Eq. 4) --------------------------------------------
        w_ent = np.exp(entropy[idx_cls] / T) if use_entropy \
            else np.ones(n_cls, dtype=float)
        if use_safe:
            w_safe = safe_ratio.copy()
            w_safe[safe_ratio < tau] = 0.0
        else:
            w_safe = np.ones(n_cls, dtype=float)
        weights = w_ent * w_safe

        n_gated = int(np.sum(safe_ratio < tau)) if use_safe else 0
        if weights.sum() <= 0:
            weights = np.ones(n_cls, dtype=float)
        prob = weights / weights.sum()

        # ---- diagnostics (item T2) --------------------------------------
        diag[int(cls)] = {
            'n_minority': n_cls,
            'H_mean': float(np.mean(entropy[idx_cls])),
            'H_median': float(np.median(entropy[idx_cls])),
            'H_std': float(np.std(entropy[idx_cls])),
            'H_frac_below_0.01': float(np.mean(entropy[idx_cls] < 0.01)),
            'S_mean': float(np.mean(safe_ratio)),
            'S_median': float(np.median(safe_ratio)),
            'S_std': float(np.std(safe_ratio)),
            'S_frac_eq_1': float(np.mean(safe_ratio >= 0.999)),
            'S_frac_below_tau': float(np.mean(safe_ratio < tau)),
            'n_gated_by_tau': n_gated,
            'W_mean': float(np.mean(weights)),
            'W_std': float(np.std(weights)),
            'W_cv': float(np.std(weights) / (np.mean(weights) + 1e-12)),
            'W_max_over_min': float(weights.max() / (weights[weights > 0].min()
                                                     + 1e-12))
            if np.any(weights > 0) else 0.0,
            'P_effective_sample_size': float(1.0 / np.sum(prob ** 2)),
            'P_ess_ratio': float(1.0 / np.sum(prob ** 2) / n_cls),
        }

        # ---- latent-space neighbour, input-space synthesis (Eq. 5) ------
        k_loc = int(min(k, n_cls - 1))
        nn_lat_cls = NearestNeighbors(n_neighbors=k_loc + 1).fit(latent[idx_cls])
        parent_local = rng.choice(n_cls, size=num_add, p=prob)
        _, ind_loc = nn_lat_cls.kneighbors(latent[idx_cls][parent_local])
        # pick a random neighbour among the k nearest (excluding self)
        pick = rng.integers(1, ind_loc.shape[1], size=num_add)
        neighbour_local = ind_loc[np.arange(num_add), pick]

        # jitter: diagonal covariance from the MINORITY class only (Eq. 6)
        sigma_diag = np.var(X[idx_cls], axis=0) * eta
        sd = np.sqrt(np.maximum(sigma_diag, 0.0))

        xp = X[idx_cls][parent_local]
        xn = X[idx_cls][neighbour_local]
        lam = rng.uniform(0.1, 0.9, size=(num_add, 1))
        delta = rng.normal(0.0, 1.0, size=(num_add, X.shape[1])) * sd
        new_samples = xp + lam * (xn - xp) + delta

        X_res = np.vstack([X_res, new_samples])
        y_res = np.hstack([y_res, np.full(num_add, cls)])

    if apply_tomek:
        n_before = len(y_res)
        try:
            tl = TomekLinks(sampling_strategy='all')
            X_res, y_res = tl.fit_resample(X_res, y_res)
        except Exception:
            pass
        diag['tomek_removed'] = int(n_before - len(y_res))

    return X_res, y_res, diag


# ======================================================================
# 3. metrics -- ONE definition of each, used by both cohorts
# ======================================================================
def g_mean(y_true, y_pred):
    """Geometric mean of per-class sqrt(sensitivity * specificity)."""
    cm = confusion_matrix(y_true, y_pred)
    with np.errstate(divide='ignore', invalid='ignore'):
        tp = np.diag(cm).astype(float)
        fn = cm.sum(axis=1) - tp
        fp = cm.sum(axis=0) - tp
        tn = cm.sum() - tp - fn - fp
        sens = np.where((tp + fn) > 0, tp / (tp + fn + 1e-12), np.nan)
        spec = np.where((tn + fp) > 0, tn / (tn + fp + 1e-12), np.nan)
        g = np.sqrt(sens * spec)
    g = g[np.isfinite(g)]
    if len(g) == 0:
        return 0.0
    return float(np.exp(np.mean(np.log(g + 1e-12))))   # geometric mean


def multiclass_brier(y_true, y_proba, classes):
    """Mean squared error between one-hot labels and predicted probabilities."""
    if y_proba is None:
        return np.nan
    Y = np.zeros_like(y_proba)
    for j, c in enumerate(classes):
        Y[:, j] = (y_true == c).astype(float)
    return float(np.mean(np.sum((y_proba - Y) ** 2, axis=1)))


def evaluate(y_true, y_pred, y_proba, classes):
    m = {
        'F1-Macro': f1_score(y_true, y_pred, average='macro'),
        'BalancedAccuracy': balanced_accuracy_score(y_true, y_pred),
        'G-Mean': g_mean(y_true, y_pred),
        # Degenerate-prediction guard. A model that emits a single class still
        # produces a finite F1, so it is silently averaged into the aggregates
        # unless it is flagged. PriorCorrection + AdaBoost does exactly this on
        # the UCI cohort (F1 = 0.05, G-Mean = 0.0).
        'n_pred_classes': int(len(np.unique(y_pred))),
        'degenerate': bool(len(np.unique(y_pred)) < len(classes)),
    }
    if y_proba is not None and y_proba.shape[1] == len(classes):
        try:
            if len(classes) == 2:
                m['AUC'] = roc_auc_score(y_true, y_proba[:, 1])
                m['AUPRC'] = average_precision_score(y_true, y_proba[:, 1])
                m['Brier'] = brier_score_loss(y_true, y_proba[:, 1])
            else:
                yb = label_binarize(y_true, classes=classes)
                m['AUC'] = roc_auc_score(yb, y_proba, multi_class='ovr',
                                         average='macro')
                m['AUPRC'] = average_precision_score(yb, y_proba,
                                                     average='macro')
                m['Brier'] = multiclass_brier(y_true, y_proba, classes)
        except Exception:
            m['AUC'] = m['AUPRC'] = m['Brier'] = np.nan
    else:
        m['AUC'] = m['AUPRC'] = m['Brier'] = np.nan
    return m


def predict_proba_safe(clf, X, n_classes):
    """Probabilities where available. Returns (proba, is_calibratable).

    LinearSVC/SGD-hinge have no predict_proba; a sigmoid/softmax of the
    decision function is returned for ranking metrics but flagged as NOT
    calibratable, so Brier scores from those models can be excluded.
    """
    if hasattr(clf, 'predict_proba'):
        try:
            p = clf.predict_proba(X)
            if p.shape[1] == n_classes:
                return p, True
        except Exception:
            pass
    if hasattr(clf, 'decision_function'):
        try:
            d = clf.decision_function(X)
            if d.ndim == 1:
                p1 = 1.0 / (1.0 + np.exp(-d))
                return np.column_stack([1 - p1, p1]), False
            e = np.exp(d - d.max(axis=1, keepdims=True))
            return e / e.sum(axis=1, keepdims=True), False
        except Exception:
            return None, False
    return None, False


# ======================================================================
# 4. classifiers
# ======================================================================
CLASSIFIERS = ["GBM", "RF", "Bagging", "XGBoost", "ExtraTrees", "AdaBoost",
               "DT", "SVM-RBF", "KNN", "SGD", "LinearSVM", "LogReg",
               "LDA", "QDA", "GNB"]


def build_classifier(name, seed, class_weight=None):
    cw = class_weight  # None or 'balanced'
    grids = {
        "GBM": (GradientBoostingClassifier(random_state=seed),
                {'n_estimators': [100, 200]}),
        "RF": (RandomForestClassifier(random_state=seed, n_jobs=-1,
                                      class_weight=cw),
               {'n_estimators': [100, 200]}),
        "Bagging": (BaggingClassifier(random_state=seed, n_jobs=-1),
                    {'n_estimators': [10, 30]}),
        "XGBoost": (XGBClassifier(eval_metric='mlogloss', random_state=seed,
                                  n_jobs=-1, verbosity=0),
                    {'n_estimators': [100, 200]}),
        "ExtraTrees": (ExtraTreesClassifier(random_state=seed, n_jobs=-1,
                                            class_weight=cw),
                       {'n_estimators': [100, 200]}),
        "AdaBoost": (AdaBoostClassifier(random_state=seed),
                     {'n_estimators': [50, 100]}),
        "DT": (DecisionTreeClassifier(random_state=seed, class_weight=cw),
               {'max_depth': [None, 10]}),
        "SVM-RBF": (SVC(kernel='rbf', probability=True, random_state=seed,
                        class_weight=cw), {'C': [1, 10]}),
        "KNN": (KNeighborsClassifier(n_jobs=-1), {'n_neighbors': [3, 5]}),
        "SGD": (SGDClassifier(random_state=seed, n_jobs=-1, loss='log_loss',
                              class_weight=cw), {'alpha': [1e-4, 1e-3]}),
        "LinearSVM": (LinearSVC(random_state=seed, max_iter=3000,
                                class_weight=cw), {'C': [1, 10]}),
        "LogReg": (LogisticRegression(random_state=seed, n_jobs=-1,
                                      max_iter=3000, class_weight=cw),
                   {'C': [1, 10]}),
        "LDA": (LinearDiscriminantAnalysis(), {}),
        "QDA": (QuadraticDiscriminantAnalysis(), {}),
        "GNB": (GaussianNB(), {}),
    }
    return grids[name]


def fit_classifier(name, X, y, seed, class_weight=None, cv=3):
    clf, grid = build_classifier(name, seed, class_weight)
    if grid:
        gs = GridSearchCV(clf, grid, cv=cv, n_jobs=-1, scoring='f1_macro')
        gs.fit(X, y)
        return gs.best_estimator_
    return clone(clf).fit(X, y)


# ======================================================================
# 5. resampling / baseline strategies
# ======================================================================
def ablation_grid(include_full_grid=True):
    """(use_entropy, use_safe, apply_tomek) keyed by display name.

    The 2x2x2 design makes the entropy axis identifiable, which the original
    4-variant grid did not (use_entropy was True in every variant).
    """
    base = {
        "EWDDBS (Uniform)":            (False, False, False),
        "EWDDBS (Uniform+Tomek)":      (False, False, True),
        "EWDDBS (Entropy)":            (True,  False, False),
        "EWDDBS (Entropy+Tomek)":      (True,  False, True),
        "EWDDBS (Safe)":               (False, True,  False),
        "EWDDBS (Safe+Tomek)":         (False, True,  True),
        "EWDDBS (Entropy+Safe)":       (True,  True,  False),
        "EWDDBS (Entropy+Safe+Tomek)": (True,  True,  True),
    }
    if include_full_grid:
        return base
    return {k: v for k, v in base.items()
            if k in ("EWDDBS (Entropy)", "EWDDBS (Entropy+Tomek)",
                     "EWDDBS (Safe)", "EWDDBS (Entropy+Safe+Tomek)")}


def prior_correction(y_proba, train_prior):
    """Threshold-moving baseline (Buda et al., 2018): divide by the training
    prior and renormalise, then take argmax."""
    p = y_proba / (train_prior[None, :] + 1e-12)
    return p / p.sum(axis=1, keepdims=True)


# ======================================================================
# 6. artefact logging
# ======================================================================
class ArtefactWriter:
    """Persists everything the post-hoc analysis needs.

    <outdir>/
        results.csv                     one row per (seed, fold, method, clf)
        diagnostics.csv                 H / S / W statistics per (seed, fold, variant)
        proba/seed{S}_fold{F}.npz       y_true + y_proba for every method x clf
        guide/seed{S}_fold{F}.npz       entropy, latent, y_train
    """

    def __init__(self, outdir):
        self.outdir = outdir
        os.makedirs(os.path.join(outdir, 'proba'), exist_ok=True)
        os.makedirs(os.path.join(outdir, 'guide'), exist_ok=True)
        self.rows = []
        self.diags = []
        self._proba = {}

    def save_guide(self, seed, fold, entropy, latent, y_train):
        np.savez_compressed(
            os.path.join(self.outdir, 'guide', f'seed{seed}_fold{fold}.npz'),
            entropy=entropy, latent=latent, y_train=y_train)

    def add_diag(self, seed, fold, method, diag):
        for cls, d in diag.items():
            if cls == 'tomek_removed':
                continue
            row = {'Seed': seed, 'Fold': fold, 'Method': method, 'Class': cls}
            row.update(d)
            self.diags.append(row)

    def add_proba(self, key, y_proba):
        if y_proba is not None:
            self._proba[key] = np.asarray(y_proba, dtype=np.float32)

    def add_result(self, **kw):
        self.rows.append(kw)

    def flush_fold(self, seed, fold, y_true):
        path = os.path.join(self.outdir, 'proba',
                            f'seed{seed}_fold{fold}.npz')
        np.savez_compressed(path, y_true=np.asarray(y_true), **self._proba)
        self._proba = {}

    def close(self):
        df = pd.DataFrame(self.rows)
        df.to_csv(os.path.join(self.outdir, 'results.csv'), index=False)
        dd = pd.DataFrame(self.diags)
        dd.to_csv(os.path.join(self.outdir, 'diagnostics.csv'), index=False)
        with open(os.path.join(self.outdir, 'config.json'), 'w') as f:
            json.dump({'T': T_TEMP, 'tau': TAU, 'k': K_NN,
                       'eta': ETA_JITTER}, f, indent=2)
        with open(os.path.join(self.outdir, 'environment.json'), 'w') as f:
            json.dump(environment_record(), f, indent=2)
        return df, dd


def environment_record():
    """Package versions and accelerator state, recorded with every run.

    A result archive that does not carry its own environment cannot support a
    claim about which versions produced it. Recording this at write time costs
    nothing and removes the need to remember it afterwards.
    """
    import importlib
    import platform
    import sys

    packages = ['tensorflow', 'sklearn', 'imblearn', 'xgboost', 'numpy',
                'pandas', 'scipy', 'statsmodels', 'pywt', 'wfdb',
                'scikit_posthocs']
    versions = {}
    for name in packages:
        try:
            versions[name] = getattr(importlib.import_module(name),
                                     '__version__', 'unknown')
        except Exception:                                   # noqa: BLE001
            versions[name] = 'not installed'

    rec = {'python': sys.version.split()[0],
           'platform': platform.platform(),
           'packages': versions,
           'gpu': [], 'tf_deterministic_ops': os.environ.get(
               'TF_DETERMINISTIC_OPS', 'unset')}
    try:
        import tensorflow as tf
        rec['gpu'] = [d.name for d in tf.config.list_physical_devices('GPU')]
    except Exception:                                       # noqa: BLE001
        pass
    return rec

In [ ]:
%%writefile run_ctu_uhb.py
"""
run_ctu_uhb.py
==============
EW-DDBS benchmark on the CTU-UHB intrapartum cohort (PhysioNet, n <= 552).

Changes relative to the original notebook:
  * the 62-dimensional feature pipeline described in Contribution (2) is now
    ACTUALLY IMPLEMENTED (ctg_features.py): descriptive stats, db4 wavelet
    decomposition, VLF/LF/HF + spectral edge frequency, Sample/Approximate/
    Permutation entropy, CTG morphology, and FHR-UC coupling. The old notebook
    produced 30 plain statistics and none of these families.
  * PCA is OFF by default, so "synthesis occurs in the original input space"
    (Contribution 1) is true here as well. Set --pca to restore the old
    behaviour, but then that claim must be qualified in the manuscript.
  * the CNN guide, jitter, neighbour search, ablation grid and G-Mean are
    imported from ewddbs_core, so both cohorts are genuinely identical
    (Contribution 3 becomes true).
  * artefacts (entropy, latent, safe-ratio stats, y_proba) are persisted.

Usage
-----
    python run_ctu_uhb.py --seeds 42 7 2024 --outdir results_ctu
    python run_ctu_uhb.py --permute-features --seeds 42 --outdir results_ctu_perm
"""

import argparse
import os
import time

import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from imblearn.over_sampling import (ADASYN, SMOTE, BorderlineSMOTE,
                                    KMeansSMOTE, RandomOverSampler, SVMSMOTE)

import ewddbs_core as C
from ctg_features import FEATURE_NAMES, extract_62_features

PH_THRESHOLD = 7.15
CACHE = 'ctu_uhb_features62_cache.npz'


def parse_ph(header_comments):
    for line in header_comments:
        parts = line.strip().lstrip('#').strip().split()
        if len(parts) >= 2 and parts[0].lower() == 'ph':
            try:
                return float(parts[1])
            except ValueError:
                continue
    return None


def prepare_ctu_uhb(max_records=552, cache=CACHE):
    """Download CTU-UHB and build the 62-D feature matrix (cached)."""
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        X, y = d['X'], d['y']
        print(f'Loaded cache: {X.shape[0]} samples, {X.shape[1]} features')
        if X.shape[1] != 62:
            raise RuntimeError(
                f'Cache holds {X.shape[1]} features, expected 62. Delete '
                f'{cache} so it is rebuilt with the new extractor.')
        print(f'Class distribution: '
              f'{dict(zip(*np.unique(y, return_counts=True)))}')
        return X, y

    import wfdb
    print('Fetching record list from PhysioNet (ctu-uhb-ctgdb/1.0.0) ...')
    rec_ids = wfdb.get_record_list('ctu-uhb-ctgdb/1.0.0')
    print(f'  -> {len(rec_ids)} records')

    feats, labels, n_no_ph, n_err = [], [], 0, 0
    t0 = time.time()
    for i, rid in enumerate(rec_ids):
        if len(feats) >= max_records:
            break
        try:
            hdr = wfdb.rdheader(rid, pn_dir='ctu-uhb-ctgdb/1.0.0')
            ph = parse_ph(hdr.comments)
            if ph is None:
                n_no_ph += 1
                continue
            rec = wfdb.rdrecord(rid, pn_dir='ctu-uhb-ctgdb/1.0.0')
            fhr, uc = rec.p_signal[:, 0], rec.p_signal[:, 1]
            feats.append(extract_62_features(fhr, uc))
            labels.append(0 if ph >= PH_THRESHOLD else 1)
            if len(feats) % 25 == 0:
                print(f'  ...{len(feats)} processed '
                      f'({(time.time() - t0) / 60:.1f} min)')
        except Exception as e:
            n_err += 1
            if n_err <= 3:
                print(f'  skipped {rid}: {type(e).__name__}: {str(e)[:70]}')

    X = np.asarray(feats, dtype=np.float32)
    y = np.asarray(labels, dtype=int)
    if X.shape[0] == 0:
        raise RuntimeError('No CTU-UHB records could be processed.')
    np.savez_compressed(cache, X=X, y=y,
                        feature_names=np.array(FEATURE_NAMES))
    print(f'\nDataset: {X.shape[0]} samples, {X.shape[1]} features')
    print(f'Class distribution: '
          f'{dict(zip(*np.unique(y, return_counts=True)))}')
    if n_no_ph:
        print(f'  ({n_no_ph} records had no parseable pH)')
    if n_err:
        print(f'  ({n_err} records failed)')
    return X, y


#: metric keys written when a (method, classifier) pair raises. Kept in one
#: place so it cannot drift away from what C.evaluate() actually returns.
FAILED_METRICS = ['F1-Macro', 'BalancedAccuracy', 'G-Mean', 'AUC', 'AUPRC',
                  'Brier', 'n_pred_classes', 'degenerate']


def run(X, y, seeds, outdir, n_splits=10, use_pca=False, full_grid=True,
        permute=False, feature_names=None):
    os.makedirs(outdir, exist_ok=True)
    writer = C.ArtefactWriter(outdir)
    classes = np.unique(y)
    n_classes = len(classes)
    t0 = time.time()

    for seed in seeds:
        rng_master = np.random.default_rng(seed)
        order = np.arange(X.shape[1])
        if permute:
            # Negative control: destroy the feature ordering the CNN topology
            # guide is claimed to exploit. If the ranking survives unchanged,
            # the ordering carries no inductive bias.
            order = rng_master.permutation(order)
            shown = ([feature_names[i] for i in order[:6]] if feature_names
                     else order[:6].tolist())
            print(f'[seed {seed}] feature order permuted: {shown} ...')
        Xs = X[:, order]

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                              random_state=seed)
        for fold, (tr, te) in enumerate(skf.split(Xs, y), start=1):
            print(f'\n=== seed {seed} | fold {fold}/{n_splits} ===')
            X_tr_raw, X_te_raw = Xs[tr], Xs[te]
            y_tr, y_te = y[tr], y[te]

            imp = SimpleImputer(strategy='median').fit(X_tr_raw)
            X_tr, X_te = imp.transform(X_tr_raw), imp.transform(X_te_raw)
            sc = StandardScaler().fit(X_tr)
            X_tr, X_te = sc.transform(X_tr), sc.transform(X_te)
            if use_pca:
                pca = PCA(n_components=0.95, random_state=seed).fit(X_tr)
                X_tr, X_te = pca.transform(X_tr), pca.transform(X_te)
                print(f'  PCA -> {X_tr.shape[1]} components '
                      f'(NOTE: input-space interpretability claim no longer holds)')

            print('  training CNN topology guide ...')
            ent, lat = C.cnn_topology_guide(X_tr, y_tr, seed=seed)
            writer.save_guide(seed, fold, ent, lat, y_tr)

            prior = np.array([np.mean(y_tr == c) for c in classes])

            methods = {'Baseline': (X_tr, y_tr)}
            for nm, smp in {
                'RandomOverSampler': RandomOverSampler(random_state=seed),
                'SMOTE': SMOTE(random_state=seed, k_neighbors=3),
                'BorderlineSMOTE': BorderlineSMOTE(random_state=seed,
                                                   k_neighbors=3),
                'SVMSMOTE': SVMSMOTE(random_state=seed, k_neighbors=3),
                'ADASYN': ADASYN(random_state=seed, n_neighbors=3),
                'KMeansSMOTE': KMeansSMOTE(random_state=seed, k_neighbors=2,
                                           cluster_balance_threshold=0.01),
            }.items():
                try:
                    methods[nm] = smp.fit_resample(X_tr, y_tr)
                except Exception as e:
                    print(f'  {nm} failed: {str(e)[:60]}')

            for nm, (ue, us, tk) in C.ablation_grid(full_grid).items():
                Xr, yr, diag = C.ew_ddbs_resample(
                    X_tr, y_tr, ent, lat, use_entropy=ue, use_safe=us,
                    apply_tomek=tk,
                    rng=np.random.default_rng(seed * 1000 + fold))
                methods[nm] = (Xr, yr)
                writer.add_diag(seed, fold, nm, diag)

            for m_name, (Xr, yr) in methods.items():
                for c_name in C.CLASSIFIERS:
                    try:
                        clf = C.fit_classifier(c_name, Xr, yr, seed)
                        y_pred = clf.predict(X_te)
                        proba, calib = C.predict_proba_safe(clf, X_te,
                                                            n_classes)
                        met = C.evaluate(y_te, y_pred, proba, classes)
                        writer.add_proba(f'{m_name}|{c_name}', proba)
                    except Exception:
                        met = {k: np.nan for k in FAILED_METRICS}
                        calib = False
                    writer.add_result(Seed=seed, Fold=fold,
                                      Oversampling=m_name, Classifier=c_name,
                                      Calibratable=calib, **met)

            for c_name in C.CLASSIFIERS:
                try:
                    clf = C.fit_classifier(c_name, X_tr, y_tr, seed,
                                           class_weight='balanced')
                    y_pred = clf.predict(X_te)
                    proba, calib = C.predict_proba_safe(clf, X_te, n_classes)
                    met = C.evaluate(y_te, y_pred, proba, classes)
                    writer.add_proba(f'ClassWeight|{c_name}', proba)
                except Exception:
                    met = {k: np.nan for k in FAILED_METRICS}
                    calib = False
                writer.add_result(Seed=seed, Fold=fold,
                                  Oversampling='ClassWeight',
                                  Classifier=c_name, Calibratable=calib, **met)

                try:
                    clf = C.fit_classifier(c_name, X_tr, y_tr, seed)
                    proba, calib = C.predict_proba_safe(clf, X_te, n_classes)
                    if proba is None:
                        raise ValueError
                    pc = C.prior_correction(proba, prior)
                    y_pred = classes[np.argmax(pc, axis=1)]
                    met = C.evaluate(y_te, y_pred, pc, classes)
                    writer.add_proba(f'PriorCorrection|{c_name}', pc)
                except Exception:
                    met = {k: np.nan for k in FAILED_METRICS}
                    calib = False
                writer.add_result(Seed=seed, Fold=fold,
                                  Oversampling='PriorCorrection',
                                  Classifier=c_name, Calibratable=calib, **met)

            writer.flush_fold(seed, fold, y_te)
            print(f'  fold done ({(time.time() - t0) / 60:.1f} min elapsed)')

    df, dd = writer.close()
    print(f'\nSaved {len(df)} result rows and {len(dd)} diagnostic rows '
          f'to {outdir}/')
    print(f'Total runtime: {(time.time() - t0) / 60:.1f} min')
    return df, dd


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--seeds', type=int, nargs='+', default=[42])
    ap.add_argument('--outdir', default='results_ctu')
    ap.add_argument('--folds', type=int, default=10)
    ap.add_argument('--max-records', type=int, default=552)
    ap.add_argument('--pca', action='store_true',
                    help='apply PCA(95%%) as in the original notebook; note '
                         'this invalidates the input-space interpretability claim')
    ap.add_argument('--reduced-grid', action='store_true')
    ap.add_argument('--permute-features', action='store_true',
                    help='randomly permute the input feature order (negative '
                         'control for the inductive-bias claim of Sec. 2.1.1; '
                         'mirrors run_uci.py so both cohorts have it)')
    a = ap.parse_args()

    X, y = prepare_ctu_uhb(max_records=a.max_records)
    run(X, y, a.seeds, a.outdir, n_splits=a.folds, use_pca=a.pca,
        full_grid=not a.reduced_grid, permute=a.permute_features,
        feature_names=FEATURE_NAMES)

In [ ]:
%%writefile analyze_results.py
"""
analyze_results.py
==================
Post-hoc analysis of an EW-DDBS run. Produces everything the revision needs:

  T2  weighting diagnostics -- is W_i degenerate?          -> diag_summary.csv
  T1  calibration (Brier, reliability curves)              -> calibration.csv
                                                              reliability_*.png
  --  aggregate tables (mean +/- SD)                       -> table_methods.csv
                                                              table_by_classifier.csv
  --  Friedman (block=fold and block=fold x classifier)
      + Nemenyi post-hoc                                   -> nemenyi_*.csv
  T7  Wilcoxon ablation contrasts                          -> ablation_wilcoxon.csv
  T6  effect sizes (Cliff's delta, Cohen's d), CIs,
      Holm-Bonferroni correction                           -> included above
  --  LaTeX snippets for the manuscript                    -> latex_tables.tex

No TensorFlow required.

Usage
-----
    python analyze_results.py --dir results_uci --ref "EWDDBS (Entropy+Safe+Tomek)"
"""

import argparse
import glob
import os

import numpy as np
import pandas as pd
from scipy import stats

METRICS = ['F1-Macro', 'G-Mean', 'BalancedAccuracy', 'AUC', 'AUPRC', 'Brier']


# ----------------------------------------------------------------------
# effect sizes and corrections
# ----------------------------------------------------------------------
def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((a[:, None] > b[None, :]).sum(axis=1))
    lt = sum((a[:, None] < b[None, :]).sum(axis=1))
    return float((gt - lt) / (len(a) * len(b)))


def cohens_d_paired(a, b):
    d = np.asarray(a) - np.asarray(b)
    return float(np.mean(d) / (np.std(d, ddof=1) + 1e-12))


def ci_mean_diff(a, b, alpha=0.05):
    d = np.asarray(a) - np.asarray(b)
    n = len(d)
    se = np.std(d, ddof=1) / np.sqrt(n)
    t = stats.t.ppf(1 - alpha / 2, n - 1)
    return float(np.mean(d) - t * se), float(np.mean(d) + t * se)


def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        running = max(running, val)
        adj[idx] = min(running, 1.0)
    return adj


def magnitude(delta):
    a = abs(delta)
    if a < 0.147:
        return 'negligible'
    if a < 0.33:
        return 'small'
    if a < 0.474:
        return 'medium'
    return 'large'


# ----------------------------------------------------------------------
# T2 -- weighting diagnostics
# ----------------------------------------------------------------------
def analyse_diagnostics(dd, outdir):
    if dd is None or len(dd) == 0:
        print('No diagnostics found.')
        return None
    cols = ['H_mean', 'H_median', 'H_frac_below_0.01', 'S_mean', 'S_median',
            'S_frac_eq_1', 'S_frac_below_tau', 'n_gated_by_tau', 'W_cv',
            'P_ess_ratio']
    cols = [c for c in cols if c in dd.columns]
    summ = (dd.groupby(['Method', 'Class'])[cols]
              .agg(['mean', 'std']).round(4))
    summ.to_csv(os.path.join(outdir, 'diag_summary.csv'))

    print('\n' + '=' * 78)
    print('T2  WEIGHTING DIAGNOSTICS  (is W_i degenerate?)')
    print('=' * 78)
    print(summ.to_string())

    full = dd[dd['Method'].str.contains('Entropy\\+Safe', regex=True)]
    if len(full) == 0:
        full = dd
    w_cv = full['W_cv'].mean() if 'W_cv' in full else np.nan
    ess = full['P_ess_ratio'].mean() if 'P_ess_ratio' in full else np.nan
    s1 = full['S_frac_eq_1'].mean() if 'S_frac_eq_1' in full else np.nan
    gated = full['S_frac_below_tau'].mean() if 'S_frac_below_tau' in full else np.nan
    h0 = full['H_frac_below_0.01'].mean() if 'H_frac_below_0.01' in full else np.nan

    print('\nVERDICT')
    print(f'  mean CV(W_i)                 = {w_cv:.4f}')
    print(f'  mean effective-sample ratio  = {ess:.4f}   (1.0 = uniform)')
    print(f'  mean fraction with S(z) = 1  = {s1:.4f}')
    print(f'  mean fraction gated by tau   = {gated:.4f}')
    print(f'  mean fraction with H < 0.01  = {h0:.4f}')
    if w_cv < 0.05 or ess > 0.98:
        print('  --> DEGENERATE: the weighting is effectively uniform, so')
        print('      EW-DDBS reduces to SMOTE with latent-space neighbours.')
        print('      Report this explicitly and consider out-of-fold H/z.')
    else:
        print('  --> NON-DEGENERATE: the weighting genuinely reshapes the')
        print('      parent distribution. Report the distributions in the paper.')
    return summ


# ----------------------------------------------------------------------
# T1 -- calibration
# ----------------------------------------------------------------------
def analyse_calibration(df, resdir, outdir, n_bins=10):
    files = sorted(glob.glob(os.path.join(resdir, 'proba', '*.npz')))
    if not files:
        print('\nNo probability files found; skipping reliability curves.')
        return None

    rows = []
    for f in files:
        d = np.load(f)
        y_true = d['y_true']
        classes = np.unique(y_true)
        for key in d.files:
            if key == 'y_true':
                continue
            p = d[key]
            if p.ndim != 2 or p.shape[0] != len(y_true):
                continue
            method, clf = key.split('|', 1)
            # positive class = last (minority in both cohorts)
            pos = p[:, -1]
            yb = (y_true == classes[-1]).astype(float)
            bins = np.clip(np.digitize(pos, np.linspace(0, 1, n_bins + 1)) - 1,
                           0, n_bins - 1)
            for b in range(n_bins):
                m = bins == b
                if m.sum() == 0:
                    continue
                rows.append({'File': os.path.basename(f), 'Method': method,
                             'Classifier': clf, 'bin': b,
                             'conf': float(pos[m].mean()),
                             'acc': float(yb[m].mean()), 'n': int(m.sum())})
    rel = pd.DataFrame(rows)
    rel.to_csv(os.path.join(outdir, 'reliability_bins.csv'), index=False)

    cal = df[df.get('Calibratable', True) == True] if 'Calibratable' in df \
        else df
    tab = (cal.groupby('Oversampling')['Brier']
              .agg(['mean', 'std', 'count']).round(4)
              .sort_values('mean'))
    tab.to_csv(os.path.join(outdir, 'calibration.csv'))

    print('\n' + '=' * 78)
    print('T1  CALIBRATION  (lower Brier is better; uncalibratable models excluded)')
    print('=' * 78)
    print(tab.to_string())

    # expected calibration error per method
    if rel.empty or 'Method' not in rel.columns:
        # No reliability bins were produced. This happens when no probability
        # file was written, for instance on a subset run with a single
        # uncalibratable classifier. Report the absence rather than crashing.
        print('\nExpected Calibration Error: not computed '
              '(no reliability bins were produced).')
        pd.DataFrame(columns=['Method', 'ECE']).to_csv(
            os.path.join(outdir, 'ece.csv'), index=False)
        return tab

    ece = (rel.groupby(['Method'])
              .apply(lambda g: np.average(np.abs(g['conf'] - g['acc']),
                                          weights=g['n']))
              .round(4).sort_values())
    ece.name = 'ECE'
    ece.to_csv(os.path.join(outdir, 'ece.csv'))
    print('\nExpected Calibration Error (minority class):')
    print(ece.to_string())
    return tab


# ----------------------------------------------------------------------
# aggregate tables + statistics
# ----------------------------------------------------------------------
def report_degenerate(df, outdir):
    """List (method, classifier) pairs that collapse to a single predicted
    class. These still yield a finite F1 and would otherwise be averaged into
    the aggregates unnoticed."""
    if 'degenerate' not in df.columns:
        return None
    g = (df.groupby(['Oversampling', 'Classifier'])['degenerate']
           .mean().reset_index())
    bad = g[g['degenerate'] > 0].sort_values('degenerate', ascending=False)
    bad.to_csv(os.path.join(outdir, 'degenerate_predictions.csv'), index=False)
    print('\n' + '=' * 78)
    print('DEGENERATE PREDICTIONS (fraction of folds collapsing to one class)')
    print('=' * 78)
    if len(bad) == 0:
        print('  none')
    else:
        print(bad.to_string(index=False))
        print('\n  Report these explicitly, or state that the aggregate is')
        print('  computed with them excluded -- do not average them in silently.')
    return bad


def aggregate(df, outdir):
    have = [m for m in METRICS if m in df.columns]
    t = (df.groupby('Oversampling')[have].agg(['mean', 'std']).round(4))
    t = t.sort_values(('F1-Macro', 'mean'), ascending=False)
    t.to_csv(os.path.join(outdir, 'table_methods.csv'))
    print('\n' + '=' * 78)
    print('AGGREGATE PERFORMANCE (mean +/- SD over seeds x folds x classifiers)')
    print('=' * 78)
    print(t.to_string())

    bc = (df.groupby(['Oversampling', 'Classifier'])[have].mean().round(4))
    bc.to_csv(os.path.join(outdir, 'table_by_classifier.csv'))
    return t


def friedman_nemenyi(df, outdir, metric='F1-Macro', ref=None):
    import scikit_posthocs as sp
    out = {}
    for scheme, keys in [('S1_fold', ['Seed', 'Fold']),
                         ('S2_fold_x_clf', ['Seed', 'Fold', 'Classifier'])]:
        piv = (df.groupby(keys + ['Oversampling'])[metric].mean()
                 .unstack().dropna())
        if piv.shape[0] < 3 or piv.shape[1] < 3:
            continue
        chi, p = stats.friedmanchisquare(*[piv[c].values for c in piv.columns])
        ranks = piv.rank(axis=1, ascending=False).mean().sort_values()
        print(f'\nFriedman [{scheme}] N={piv.shape[0]}: '
              f'chi2={chi:.3f}, p={p:.3e}')
        print('  average ranks (lower = better):')
        for k, v in ranks.items():
            print(f'    {k:32s} {v:.3f}')
        nem = sp.posthoc_nemenyi_friedman(piv.values)
        nem.index = nem.columns = piv.columns
        nem.round(4).to_csv(os.path.join(outdir, f'nemenyi_{scheme}.csv'))
        if ref and ref in nem.columns:
            s = nem[ref].drop(ref).sort_values()
            print(f'  Nemenyi vs {ref}:')
            for k, v in s.items():
                flag = '*' if v < 0.05 else 'ns'
                print(f'    {k:32s} p={v:.4f} {flag}')
        out[scheme] = {'chi2': chi, 'p': p, 'ranks': ranks, 'nemenyi': nem}
    return out


def ablation_tests(df, outdir, metrics=('F1-Macro', 'AUPRC')):
    """Wilcoxon on every EW-DDBS pair, with effect size, CI and Holm."""
    variants = sorted(v for v in df['Oversampling'].unique()
                      if v.startswith('EWDDBS'))
    rows = []
    for m in metrics:
        if m not in df.columns:
            continue
        for i in range(len(variants)):
            for j in range(i + 1, len(variants)):
                a, b = variants[i], variants[j]
                sa = (df[df['Oversampling'] == a]
                      .set_index(['Seed', 'Fold', 'Classifier'])[m])
                sb = (df[df['Oversampling'] == b]
                      .set_index(['Seed', 'Fold', 'Classifier'])[m])
                idx = sa.index.intersection(sb.index)
                sa, sb = sa.loc[idx].dropna(), sb.loc[idx].dropna()
                idx = sa.index.intersection(sb.index)
                sa, sb = sa.loc[idx].values, sb.loc[idx].values
                if len(sa) < 5 or np.all(sa == sb):
                    continue
                try:
                    stat, p = stats.wilcoxon(sa, sb, zero_method='wilcox')
                except Exception:
                    continue
                lo, hi = ci_mean_diff(sb, sa)
                d = cliffs_delta(sb, sa)
                rows.append({'Metric': m, 'A': a, 'B': b, 'N': len(sa),
                             'mean_diff_B_minus_A': round(float(np.mean(sb - sa)), 5),
                             'CI95_low': round(lo, 5), 'CI95_high': round(hi, 5),
                             'cliffs_delta': round(d, 4),
                             'magnitude': magnitude(d),
                             'cohens_d': round(cohens_d_paired(sb, sa), 4),
                             'p_raw': round(float(p), 5)})
    if not rows:
        return None
    t = pd.DataFrame(rows)
    # Holm correction WITHIN each metric family, aligned by index so the
    # adjusted value always lands on the row it belongs to.
    t['p_holm'] = np.nan
    for m, g in t.groupby('Metric'):
        t.loc[g.index, 'p_holm'] = holm(g['p_raw'].values)
    t['p_holm'] = t['p_holm'].round(5)
    t['sig_holm'] = np.where(t['p_holm'] < 0.05, '*', 'ns')
    assert (t['p_holm'] >= t['p_raw'] - 1e-9).all(), \
        'Holm-adjusted p must never be smaller than the raw p'
    t = t.sort_values(['Metric', 'p_raw'])
    t.to_csv(os.path.join(outdir, 'ablation_wilcoxon.csv'), index=False)
    print('\n' + '=' * 78)
    print('T6/T7  ABLATION CONTRASTS (Wilcoxon + effect size + Holm)')
    print('=' * 78)
    print(t.to_string(index=False))
    return t


def latex_tables(t_methods, outdir):
    path = os.path.join(outdir, 'latex_tables.tex')
    with open(path, 'w') as f:
        f.write('% auto-generated -- paste into the manuscript\n')
        f.write(t_methods.to_latex(escape=False))
    print(f'\nLaTeX snippets written to {path}')


# ----------------------------------------------------------------------
def main(resdir, ref):
    outdir = os.path.join(resdir, 'analysis')
    os.makedirs(outdir, exist_ok=True)
    df = pd.read_csv(os.path.join(resdir, 'results.csv'))
    dpath = os.path.join(resdir, 'diagnostics.csv')
    dd = pd.read_csv(dpath) if os.path.exists(dpath) else None

    analyse_diagnostics(dd, outdir)
    report_degenerate(df, outdir)
    t = aggregate(df, outdir)
    analyse_calibration(df, resdir, outdir)
    friedman_nemenyi(df, outdir, ref=ref)
    ablation_tests(df, outdir)
    latex_tables(t, outdir)
    print(f'\nAll analysis artefacts in {outdir}/')


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--dir', required=True)
    ap.add_argument('--ref', default='EWDDBS (Entropy+Safe+Tomek)')
    a = ap.parse_args()
    main(a.dir, a.ref)

In [ ]:
%%writefile merge_runs.py
"""
merge_runs.py
=============
Gabungkan beberapa direktori hasil (mis. seed 42 yang sudah jalan + seed 7 & 2024
yang baru) menjadi satu direktori siap dianalisis.

Aman dipakai karena berkas proba/ dan guide/ dinamai seed{S}_fold{F}.npz,
sehingga tidak pernah bertabrakan antar-seed.

Catatan penting untuk Google Colab
----------------------------------
/content DIHAPUS setiap kali runtime di-reset atau terputus. Jadi direktori
hasil dari sesi sebelumnya (mis. results_uci) TIDAK akan ada lagi di sesi baru,
meskipun Anda sudah mengunduh zip-nya ke komputer.

Skrip ini menanganinya dengan dua cara:
  1. Bila `results_uci/` hilang tetapi `results_uci.zip` ada di direktori kerja
     (atau di Google Drive yang ter-mount), arsip itu diekstrak otomatis.
  2. Bila benar-benar tidak ada, skrip berhenti SEBELUM membuat direktori
     keluaran, dan mencetak daftar apa saja yang sebenarnya tersedia.

Pemakaian
---------
    python merge_runs.py --out results_uci_all results_uci results_uci_s2
    python analyze_results.py --dir results_uci_all --ref "EWDDBS (Entropy+Safe+Tomek)"
"""

import argparse
import glob
import os
import shutil
import zipfile

import pandas as pd


#: tempat tambahan yang ikut dicari saat sebuah arsip .zip dilacak
ZIP_SEARCH_PATHS = [
    '.',
    '/content',
    '/content/drive/MyDrive',
    '/content/drive/MyDrive/ewddbs',
]


def _find_zip(name):
    """Cari <name>.zip di direktori kerja dan di Drive yang ter-mount."""
    for base in ZIP_SEARCH_PATHS:
        cand = os.path.join(base, f'{name}.zip')
        if os.path.exists(cand):
            return cand
    return None


def restore_if_needed(d):
    """Kembalikan direktori hasil dari arsip .zip bila direktorinya hilang.

    Ini kasus yang paling sering terjadi di Colab: runtime di-reset, /content
    kosong, tetapi pengguna masih punya results_uci.zip. Mengekstrak ulang jauh
    lebih baik daripada menjalankan ulang eksperimen 40 menit.
    """
    if os.path.isdir(d):
        return True

    z = _find_zip(os.path.basename(d.rstrip('/')))
    if z is None:
        return False

    print(f'  {d}/ tidak ada — memulihkan dari {z}')
    os.makedirs(d, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(d)

    # make_archive() mengarsipkan ISI folder, tetapi sebagian pengguna membuat
    # zip yang membungkus foldernya sekali lagi. Tangani kedua bentuk itu.
    if not os.path.exists(os.path.join(d, 'results.csv')):
        inner = os.path.join(d, os.path.basename(d))
        if os.path.exists(os.path.join(inner, 'results.csv')):
            for item in os.listdir(inner):
                shutil.move(os.path.join(inner, item), os.path.join(d, item))
            os.rmdir(inner)

    ok = os.path.exists(os.path.join(d, 'results.csv'))
    print(f'    -> {"berhasil" if ok else "GAGAL: results.csv tidak ada di dalam zip"}')
    return ok


def _inventory():
    """Daftar direktori hasil dan arsip yang benar-benar ada, untuk pesan galat."""
    dirs = sorted(p for p in glob.glob('*')
                  if os.path.isdir(p)
                  and os.path.exists(os.path.join(p, 'results.csv')))
    zips = sorted(os.path.basename(p) for base in ZIP_SEARCH_PATHS
                  for p in glob.glob(os.path.join(base, 'results*.zip')))
    lines = ['\nYang tersedia di direktori kerja saat ini:']
    lines.append('  direktori hasil : ' + (', '.join(dirs) if dirs else '(tidak ada)'))
    lines.append('  arsip zip       : ' + (', '.join(sorted(set(zips))) if zips
                                           else '(tidak ada)'))
    lines.append('')
    lines.append('Jika ini sesi Colab yang baru, /content sudah dikosongkan.')
    lines.append('Unggah kembali zip hasil sesi sebelumnya lewat panel Files')
    lines.append('(ikon folder di kiri), lalu jalankan ulang sel ini — skrip')
    lines.append('akan mengekstraknya sendiri.')
    return '\n'.join(lines)


def validate(dirs):
    """Pastikan SEMUA direktori masukan ada sebelum apa pun ditulis."""
    missing = []
    for d in dirs:
        if not restore_if_needed(d):
            missing.append(d)
        elif not os.path.exists(os.path.join(d, 'results.csv')):
            missing.append(d)
    if missing:
        raise FileNotFoundError(
            'Direktori hasil berikut tidak ditemukan dan tidak ada arsipnya: '
            + ', '.join(missing) + '\n' + _inventory())


def merge(out, dirs):
    # Validasi dijalankan LEBIH DULU. Versi sebelumnya membuat direktori
    # keluaran sebelum memeriksa masukan, sehingga saat merge gagal ia
    # meninggalkan results_uci_all/ yang kosong — dan perintah analyze
    # berikutnya melaporkan galat kedua yang menyesatkan.
    validate(dirs)

    os.makedirs(os.path.join(out, 'proba'), exist_ok=True)
    os.makedirs(os.path.join(out, 'guide'), exist_ok=True)

    res, diag = [], []
    for d in dirs:
        r = pd.read_csv(os.path.join(d, 'results.csv'))
        res.append(r)
        print(f'  {d:24s} {len(r):6d} baris | seed {sorted(r.Seed.unique())}')

        dp = os.path.join(d, 'diagnostics.csv')
        if os.path.exists(dp):
            diag.append(pd.read_csv(dp))

        for sub in ('proba', 'guide'):
            src = os.path.join(d, sub)
            if not os.path.isdir(src):
                continue
            for f in os.listdir(src):
                dst = os.path.join(out, sub, f)
                if os.path.exists(dst):
                    print(f'    ! {sub}/{f} sudah ada — dilewati '
                          f'(seed yang sama dijalankan dua kali?)')
                    continue
                shutil.copy2(os.path.join(src, f), dst)

    R = pd.concat(res, ignore_index=True)
    dup = R.duplicated(subset=['Seed', 'Fold', 'Oversampling', 'Classifier'])
    if dup.any():
        print(f'\n! {int(dup.sum())} baris duplikat (seed x fold x metode x '
              f'classifier) dibuang')
        R = R[~dup]
    R.to_csv(os.path.join(out, 'results.csv'), index=False)

    if diag:
        D = pd.concat(diag, ignore_index=True)
        D = D.drop_duplicates(subset=['Seed', 'Fold', 'Method', 'Class'])
        D.to_csv(os.path.join(out, 'diagnostics.csv'), index=False)
    else:
        D = pd.DataFrame()

    for d in dirs:
        cp = os.path.join(d, 'config.json')
        if os.path.exists(cp):
            shutil.copy2(cp, os.path.join(out, 'config.json'))
            break

    print(f'\nGabungan -> {out}/')
    print(f'  results.csv     : {len(R)} baris, seed {sorted(R.Seed.unique())}')
    print(f'  diagnostics.csv : {len(D)} baris')
    print(f'  proba/          : {len(os.listdir(os.path.join(out, "proba")))} berkas')
    print(f'  guide/          : {len(os.listdir(os.path.join(out, "guide")))} berkas')

    n_exp = R.Seed.nunique() * R.Fold.nunique() * R.Oversampling.nunique() * \
        R.Classifier.nunique()
    print(f'  kelengkapan     : {len(R)}/{n_exp} '
          f'({100 * len(R) / n_exp:.1f}%)')
    if len(R) < n_exp:
        print('  ! tidak lengkap — periksa apakah ada seed/fold yang gagal')
    return R, D


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', required=True)
    ap.add_argument('dirs', nargs='+')
    a = ap.parse_args()
    merge(a.out, a.dirs)

## 3. Uji cepat ekstraktor fitur

In [ ]:
!python ctg_features.py

## 4. Simpan cache ke Drive

In [ ]:
import os, shutil
if os.path.exists(CACHE) and os.path.isdir(DRIVE):
    shutil.copy(CACHE, f'{DRIVE}/{CACHE}'); print('cache disalin ke Drive')
else:
    print('dilewati (cache atau Drive tidak tersedia)')

## 5. Seed 42 — SUDAH SELESAI, JANGAN DIJALANKAN ULANG

Angka seed 42 sudah masuk Tabel 4-8 naskah. Karena komponen CNN berjalan
tanpa determinisme, menjalankan ulang seed yang sama menghasilkan angka
**berbeda**, dan seluruh tabel CTU harus dihitung ulang.

Bila folder hilang karena runtime di-reset, unggah `results_ctu.zip` —
`merge_runs.py` mengekstraknya otomatis di Sel 8.

In [ ]:
import os

if os.path.exists('results_ctu/results.csv'):
    print('results_ctu/ sudah ada - dilewati (benar).')
elif os.path.exists('results_ctu.zip'):
    print('results_ctu.zip ditemukan - akan dipulihkan otomatis saat merge.')
else:
    print('PERINGATAN: hasil seed 42 tidak ditemukan.')
    print('Unggah results_ctu.zip, ATAU hapus komentar baris di bawah dan')
    print('terima bahwa SEMUA angka CTU di naskah harus dihitung ulang.')
    # !python run_ctu_uhb.py --seeds 42 --outdir results_ctu --folds 10

## 6. Seed 7  (~85 menit)

Jalankan sendirian; jangan digabung dengan Sel 7.

In [ ]:
!python run_ctu_uhb.py --seeds 7 --outdir results_ctu_s2 --folds 10

## 7. Seed 2024  (~85 menit)

In [ ]:
!python run_ctu_uhb.py --seeds 2024 --outdir results_ctu_s3 --folds 10

## 8. Cek kelengkapan, gabungkan, analisis

In [ ]:
import os, glob

for name in ['results_ctu', 'results_ctu_s2', 'results_ctu_s3', 'results_ctu_perm']:
    has_dir = os.path.exists(os.path.join(name, 'results.csv'))
    has_zip = os.path.exists(f'{name}.zip')
    print(f"{'OK' if has_dir else ('ZIP' if has_zip else 'HILANG'):7s} {name}")
print('\nZip di direktori kerja:', glob.glob('results*.zip') or '(tidak ada)')

In [ ]:
!python merge_runs.py --out results_ctu_all results_ctu results_ctu_s2 results_ctu_s3 && \
 python analyze_results.py --dir results_ctu_all --ref "EWDDBS (Entropy+Safe+Tomek)"

## 9. Kontrol permutasi — SUDAH SELESAI (rujukan saja)

In [ ]:
# !python run_ctu_uhb.py --permute-features --seeds 42 --outdir results_ctu_perm --folds 10
# !python analyze_results.py --dir results_ctu_perm --ref "EWDDBS (Entropy+Safe+Tomek)"

## 10. Unduh semua hasil

In [ ]:
import shutil, os

for name in ['results_ctu', 'results_ctu_s2', 'results_ctu_s3',
             'results_ctu_all', 'results_ctu_perm']:
    if not os.path.isdir(name):
        print(f'{name:22s} (tidak ada - dilewati)')
        continue
    shutil.make_archive(name, 'zip', name)
    print(f'{name:22s} -> {name}.zip  ({os.path.getsize(name + ".zip") / 1e6:.1f} MB)')
    try:
        from google.colab import files
        files.download(f'{name}.zip')
    except Exception as e:
        print('   (bukan Colab / unduh manual:', e, ')')